# Setting

## Module installation and device setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import lr_scheduler

from pyproj import Transformer
import math

import itertools

import numpy as np
import pandas as pd
import pickle as pickle
import random
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression

import os
import shutil
import rasterio
from PIL import Image
from scipy import ndimage

#### set device ####
random.seed(42)
np.random.seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

c:\coding\bong38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda:0


# Construct Model and modeling function

## Construct integrated SSL-DL models

In [2]:
#### pretext task ####
class CNNAE(nn.Module):

    def __init__(self):
        super(CNNAE, self).__init__()
        self.conv1encoder = nn.Sequential(
            nn.Conv1d(in_channels = in_ch,                            
                      out_channels = out_ch1,                   
                      kernel_size = 1,  
                      ),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch1,
                      out_channels = out_ch2,
                      kernel_size = 1,
                      ),
            nn.BatchNorm1d(num_features=out_ch2,eps= 1e-05, momentum=0.1),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch2,
                      out_channels = out_ch3,
                      kernel_size = 1,
                      ),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch3,
                      out_channels = out_ch4,
                      kernel_size = 1,
                      ),
            nn.BatchNorm1d(num_features=out_ch4,eps= 1e-05, momentum=0.1),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch4,
                      out_channels = out_ch5,
                      kernel_size = 1,
                      ),
            nn.LeakyReLU()
            )

        self.conv1decoder1 = nn.Sequential(
            nn.ConvTranspose1d(in_channels = out_ch5,                           
                               out_channels = out_ch4,                       
                               kernel_size = 1,              
                               ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch4,
                               out_channels = out_ch3,
                               kernel_size = 1,
                               ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch3,
                               out_channels = in_ch,
                               kernel_size = 1,
                               ),
            nn.LeakyReLU()
            )
        
        self.conv1decoder2 = nn.Sequential(
            nn.ConvTranspose1d(in_channels = out_ch5,                          
                      out_channels = out_ch4,                       
                      kernel_size = 1,               
                      ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch4,
                      out_channels = out_ch3,
                      kernel_size = 1,
                      # stride = 1,
                      # padding = 0
                      ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch3,
                      out_channels = out_ch6,  
                      kernel_size = 1,
                      ),
            nn.LeakyReLU()
            )

    def forward(self, x1):

        encoded=self.conv1encoder(x1)
        decoded1=self.conv1decoder1(encoded)
        decoded2=self.conv1decoder2(encoded)

        return encoded, decoded1, decoded2

#### downstream task ####
class MLP_base(nn.Module):

    def __init__(self, original_model):

        super(MLP_base, self).__init__()

        self.original = nn.Sequential(*list(original_model.conv1encoder))

        self.fc1 = nn.Linear(in_ch_m,                            
                      out_ch2                      
                      )

        self.fc2= nn.Linear(out_ch2,                            
                      out_ch3                       
                      )
        self.fc3= nn.Linear(out_ch3,                            
                      1                         
                      )
        self.dropout=nn.Dropout(do)
        self.LeakyReLU=nn.LeakyReLU()

    def forward(self, x):

        x = self.original(x)
        x = x.squeeze(0)
        x = x.transpose(0, 1)
        out1 = self.LeakyReLU(self.fc1(x))
        out2 = self.LeakyReLU(self.dropout(self.fc2(out1)))
        out3 =  self.LeakyReLU(self.fc3(out2))

        return out3
    
## Generative
#### pretext task ####
class CNNAE_no_WSL(nn.Module):

    def __init__(self):
        super(CNNAE_no_WSL, self).__init__()
        self.conv1encoder = nn.Sequential(
            nn.Conv1d(in_channels = in_ch,                           
                      out_channels = out_ch1,                       
                      kernel_size = 1,   
                      ),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch1,
                      out_channels = out_ch2,
                      kernel_size = 1,
                      ),
            nn.BatchNorm1d(num_features=out_ch2,eps= 1e-05, momentum=0.1),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch2,
                      out_channels = out_ch3,
                      kernel_size = 1,
                      ),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch3,
                      out_channels = out_ch4,
                      kernel_size = 1,
                      ),
            nn.BatchNorm1d(num_features=out_ch4,eps= 1e-05, momentum=0.1),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch4,
                      out_channels = out_ch5,
                      kernel_size = 1,
                      ),
            nn.LeakyReLU()
            )

        self.conv1decoder1 = nn.Sequential(
            nn.ConvTranspose1d(in_channels = out_ch5,                            
                               out_channels = out_ch4,                       
                               kernel_size = 1,               
                               ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch4,
                               out_channels = out_ch3,
                               kernel_size = 1,
                               ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch3,
                               out_channels = in_ch,
                               kernel_size = 1,
                               ),
            nn.LeakyReLU()
            )

    def forward(self, x1):

        encoded=self.conv1encoder(x1)
        decoded1=self.conv1decoder1(encoded)

        return encoded, decoded1

#Predictive SSL    
#### pretext task ####
class Pred_CNNAE(nn.Module):

    def __init__(self):
        super(Pred_CNNAE, self).__init__()
        self.conv1encoder = nn.Sequential(
            nn.Conv1d(in_channels = in_ch,                          
                      out_channels = out_ch1,                      
                      kernel_size = 1,          
                      ),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch1,
                      out_channels = out_ch2,
                      kernel_size = 1,
                      ),
            nn.BatchNorm1d(num_features=out_ch2,eps= 1e-05, momentum=0.1),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch2,
                      out_channels = out_ch3,
                      kernel_size = 1,
                      ),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch3,
                      out_channels = out_ch4,
                      kernel_size = 1,
                      ),
            nn.BatchNorm1d(num_features=out_ch4,eps= 1e-05, momentum=0.1),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels = out_ch4,
                      out_channels = out_ch5,
                      kernel_size = 1,
                      ),
            nn.LeakyReLU()
            )
        
        self.conv1decoder2 = nn.Sequential(
            nn.ConvTranspose1d(in_channels = out_ch5,                           
                      out_channels = out_ch4,                         
                      kernel_size = 1,              
                      ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch4,
                      out_channels = out_ch3,
                      kernel_size = 1,
                      ),
            nn.LeakyReLU(),
            nn.ConvTranspose1d(in_channels = out_ch3,
                      out_channels = out_ch6,  
                      kernel_size = 1,
                      ),
            nn.LeakyReLU()
            )

    def forward(self, x1):

        encoded=self.conv1encoder(x1)
        decoded2=self.conv1decoder2(encoded)

        return encoded, decoded2

## Performance metrics and minmax function

In [3]:
#### model performance ####
def rmse(targets, predictions):
    return np.sqrt(((predictions - targets) ** 2).mean())

def smape(true, pred):  
    return np.mean((np.abs(true-pred))/(np.abs(true) + np.abs(pred)))

def mae(targets, predictions):
    return mean_absolute_error(targets.reshape(-1, 1), predictions.reshape(-1, 1))

def nse(targets, predictions):
    return 1-(np.sum((targets-predictions)**2)/np.sum((targets-np.mean(targets))**2))

#### unlabeled data minmax function ####
def minMax(pretrain_data_tensor):
    num_channels = pretrain_data_tensor.shape[1]
    normalized_tensors = []
    scale_factors = []
    min_vals = []
    
    for i in range(num_channels):
        channel_data = pretrain_data_tensor[:, i]
        min_val = torch.min(channel_data)
        max_val = torch.max(channel_data)
        
        if min_val == max_val:
            # No scaling needed for this channel (all values are the same)
            normalized_tensor = channel_data.float()
            scale_factor = 1.0
        else:
            # Apply normalization to the channel data
            scale_factor = 1.0 / (max_val - min_val)
            normalized_tensor = (channel_data - min_val) * scale_factor
        
        normalized_tensors.append(normalized_tensor)
        scale_factors.append(scale_factor)
        min_vals.append(min_val)
        
    normalized_pretrain_data = torch.stack(normalized_tensors, dim=1)

    return   normalized_pretrain_data, scale_factors, min_vals

#### restore minmax data(label & unlabel) to original data ####
def convert_to_original(pretrain_data_tensor, scale_factors, min_vals):
    
    num_channels = pretrain_data_tensor.shape[1]
    
    original_tensors = []
    if len(scale_factors) !=1:
        for i in range(num_channels):
            channel_data = pretrain_data_tensor[:, i]
            channel_original = (channel_data / scale_factors[i]) + min_vals[i]
            original_tensors.append(channel_original)
    else:
        for i in range(num_channels):
            channel_data = pretrain_data_tensor[:, i]
            channel_original = (channel_data / scale_factors[0]) + min_vals[0]
            original_tensors.append(channel_original)
            
    pretrain_dataset_original = torch.stack(original_tensors, dim=1)
    
    return pretrain_dataset_original

## Training function

In [4]:
#### pretext task learning for unlabel ####
def train_AE(models, train_loader,optimizer,mse_loss,scale_factors,min_vals,ft_scale_factors,ft_min_vals):
    models.train()
    avg_loss=0
    for batch_idx, (x, label) in enumerate(train_loader):
        optimizer.zero_grad()
        x_ori = x.to(device)
        label_tensor=label.to(device)
        x_train_tensor=x_ori
        x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
        x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()
        
        encoded, decoded1, decoded2 = models(x_train_tensor)
        preict_ori = decoded1.squeeze(0)
        preict_ori = preict_ori.transpose(0, 1).contiguous()
        preict_pseudo = decoded2.squeeze(0)
        preict_pseudo = preict_pseudo.transpose(0, 1).contiguous()
        
        preict_ori = convert_to_original(preict_ori, scale_factors, min_vals)
        x_ori = convert_to_original(x_ori, scale_factors, min_vals)
        
        loss1=mse_loss(preict_ori,x_ori)
        loss2=mse_loss(preict_pseudo,label_tensor)
        loss=loss1+loss_ratio*loss2
        loss.backward()
        optimizer.step()

        avg_loss+=loss.item()

    return avg_loss / len(train_loader)

#### pretext task learning for label ####
def train_AE_com(models, train_loader,optimizer,mse_loss, scale_factors, min_vals,ft_scale_factors,ft_min_vals):
    models.train()
    avg_loss=0
    for batch_idx, (x_ori_p, label_ori) in enumerate(train_loader):
        optimizer.zero_grad()
        if x_ori_p.size(0)<10:
            pad_len = 10 - x_ori_p.size(0)
            padding = torch.zeros(pad_len, x_ori_p.size(1), dtype=torch.float)
            x = torch.cat([x_ori_p, padding], dim=0)
            label_padding1= torch.zeros(pad_len, label_ori.size(1))
            label = torch.cat([label_ori,label_padding1], dim=0)
        else:    
            x=x_ori_p
            label=label_ori
        
        x_ori = x.to(device)
        label_tensor=label.to(device)
        x_train_tensor=x_ori
        x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
        x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()
        
        encoded, decoded1, decoded2 = models(x_train_tensor) 
        preict_ori = decoded1.squeeze(0)
        preict_ori = preict_ori.transpose(0, 1).contiguous()
        preict_pseudo = decoded2.squeeze(0)
        preict_pseudo = preict_pseudo.transpose(0, 1).contiguous()
        
        preict_ori = convert_to_original(preict_ori, scale_factors, min_vals)
        x_ori = convert_to_original(x_ori, scale_factors, min_vals)
        mask_pad = label_tensor != 0 
        loss1=mse_loss(preict_ori[mask_pad[:,0],:],x_ori[mask_pad[:,0],:])
        loss2=mse_loss(preict_pseudo[mask_pad[:,0],:],label_tensor[mask_pad[:,0],:])
        loss=loss1+loss_ratio*loss2
        loss.backward()
        optimizer.step()

        avg_loss+=loss.item()

    return avg_loss / len(train_loader)

#### pretext task learning without WSL ####
def train_AE_no_wsl(models, train_loader,optimizer,mse_loss,scale_factors,min_vals,ft_scale_factors,ft_min_vals):
    models.train()
    avg_loss=0
    for batch_idx, (x, label) in enumerate(train_loader):
        optimizer.zero_grad()
        x_ori = x.to(device)
        label_tensor=label.to(device)
        x_train_tensor=x_ori
        x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
        x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()
        
        encoded, decoded1 = models(x_train_tensor) 
        preict_ori = decoded1.squeeze(0)
        preict_ori = preict_ori.transpose(0, 1).contiguous()
        
        preict_ori = convert_to_original(preict_ori, scale_factors, min_vals)
        x_ori = convert_to_original(x_ori, scale_factors, min_vals)
        
        loss=mse_loss(preict_ori,x_ori)
        loss.backward()
        optimizer.step()

        avg_loss+=loss.item()

    return avg_loss / len(train_loader)

#### pretext task learning Predictive ####
#### pretext task learning for unlabel ####
def train_AE_pred(models, train_loader,optimizer,mse_loss,scale_factors,min_vals,ft_scale_factors,ft_min_vals):
    models.train()
    avg_loss=0
    for batch_idx, (x, label) in enumerate(train_loader):
        optimizer.zero_grad()
        x_ori = x.to(device)
        label_tensor=label.to(device)
        x_train_tensor=x_ori
        x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
        x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()
        
        encoded, decoded2 = models(x_train_tensor) #, Rrss
        preict_pseudo = decoded2.squeeze(0)
        preict_pseudo = preict_pseudo.transpose(0, 1).contiguous()
        
        loss=mse_loss(preict_pseudo,label_tensor)
        loss.backward()
        optimizer.step()

        avg_loss+=loss.item()

    return avg_loss / len(train_loader)

#### downsteam task learning####
def train_MLP(models, train_loader,optimizer,MLP_mse_loss, scale_factors, min_vals):
    models.train()
    avg_loss=0
    pred_batches = []
    label_batches = []
    for batch_idx, (x_ori, label_ori) in enumerate(train_loader):
        optimizer.zero_grad()
        if x_ori.size(0)<10:
            pad_len = 10 - x_ori.size(0)
            padding = torch.zeros(pad_len, x_ori.size(1), dtype=torch.float)
            x = torch.cat([x_ori, padding], dim=0)
            label_padding1= torch.zeros(pad_len, label_ori.size(1))
            label = torch.cat([label_ori,label_padding1], dim=0)
        else:    
            x=x_ori
            label=label_ori
        
        x_train_tensor=x.to(device)
        x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
        x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()
        
        label_tensor = label.to(device)
        
        pred = models(x_train_tensor) #, Rrss
        pred = convert_to_original(pred, scale_factors, min_vals)
        label_tensor = convert_to_original(label_tensor, scale_factors, min_vals)        
        mask_pad = label_tensor != 0 
        pred_batches.append(pred)
        label_batches.append(label_tensor)
        MLP_loss=MLP_mse_loss(pred[mask_pad],label_tensor[mask_pad])
        MLP_loss.backward()
        optimizer.step()
        avg_loss+=MLP_loss.item()
        
    return avg_loss / len(train_loader) ,torch.cat(pred_batches, dim=0), torch.cat(label_batches, dim=0)

## Data indexing and related function setup

In [5]:
convert_psedo={'BR_564':31}
convert_lake_name={'Daecheong':'Daecheongdam','Paldang':'Paldangdam'}
convert_lake_index={'Daecheong1':'D1','Daecheong2':'D2','Daecheong3':'D3','Daecheong4':'D4', 
                    'Daecheong5':'D5', 'Daeheong6':'D6',
                    'Paldang1':'P1','Paldang2':'P2','Paldang3':'P3','Paldang4':'P4', 
                    'Paldang5':'P5', 'Paldang6':'P6'}

def station_indexing(psedo_label_list):
    index_psedo_label=[]
    for psedo_label_n in psedo_label_list:
        new_value=convert_lake_index[psedo_label_n]
        index_psedo_label.append(new_value)
    return index_psedo_label
def psedo_label_indexing(psedo_label_list):
    index_psedo_label=[]
    for psedo_label_n in psedo_label_list:
        new_value=convert_psedo[psedo_label_n]
        index_psedo_label.append(new_value)
    return index_psedo_label

# Model training

## Data loading

In [ ]:
df0=pd.read_csv("Data\total_label_data.csv", encoding='utf-8-sig')

## Training

In [ ]:
psedo_label_set=['BR_564']
lake_name ='Daecheong' # 'Paldang'


lake_name2=new_value=convert_lake_name[lake_name]
df=df0
df=df[df['lake_x']==lake_name2]
plot_size=np.max(df.iloc[:, [6]])
count0=0
with pd.option_context('mode.use_inf_as_na', True):
    df['BR_564_L']=df['band_5']-(df['band_6']+df['band_4'])/2

d_train, d_test = train_test_split(df, test_size = 0.2, stratify = df['sampling_point'], random_state = 42)

train_ix, train_x_o, train_y_o= d_train['Unnamed: 0'], d_train.iloc[:,list(range(19, 27))], d_train.iloc[:, [6]]
test_ix, test_x_o, test_y_o= d_test['Unnamed: 0'], d_test.iloc[:,list(range(19, 27)) ], d_test.iloc[:, [6]] 
spectra_names = train_x_o.columns
pig_names = train_y_o.columns

tr_M = train_x_o.max()
tr_m = train_x_o.min()

train_x = (train_x_o - tr_m)/(tr_M - tr_m)
test_x = (test_x_o - tr_m)/(tr_M - tr_m)

tr_M_y = train_y_o.max()
tr_m_y = train_y_o.min()

train_y = (train_y_o - tr_m_y)/(tr_M_y - tr_m_y)
test_y = (test_y_o - tr_m_y)/(tr_M_y - tr_m_y)

fit_data_scale=1.0 / (tr_M - tr_m)
fit_data_min=tr_m
fit_data_scale = fit_data_scale.tolist()
fit_data_min = fit_data_min.tolist()

fit_data_scale_y=1.0 / (tr_M_y - tr_m_y)
fit_data_min_y=tr_m_y
fit_data_scale_y = fit_data_scale_y.tolist()
fit_data_min_y = fit_data_min_y.tolist()

# SSL downstream task's labeled data
train_x_tensor = torch.Tensor(np.array(train_x))
train_x_tensor=train_x_tensor.float()
train_y_tensor= torch.Tensor(np.array(train_y))
train_y_tensor=train_y_tensor.float()

bsize=10
train_all =TensorDataset(train_x_tensor, train_y_tensor)
train_all_loader = DataLoader(train_all, batch_size=bsize, shuffle=True)

# SSL-L pretrain's pseudo-labeled data 
index_psedo=psedo_label_indexing(psedo_label_set)
train_pseudo = d_train.iloc[:, index_psedo]
test_pseudo=d_test.iloc[:, index_psedo]
train_p_tensor= torch.Tensor(np.array(train_pseudo))
train_p_tensor=train_p_tensor.float()

bsize=10
label_pretrain=TensorDataset(train_x_tensor, train_p_tensor)
train_P_loader = DataLoader(label_pretrain, batch_size=bsize, shuffle=True)

#### Consturct unlabeled data ####

with open('Data\total_unlabel_data.pkl','rb') as f:
    mydict=pickle.load(f)
all_pretrain_data=mydict.copy()
    
subset_lakes=[lake_name]

DN_dict0=all_pretrain_data[lake_name]
human_label = pd.read_excel('Data\\'+lake_name2 +'_cloud.xlsx')
human_label01=human_label['file_name'][(human_label['manual']==0) | (human_label['manual']==1)].to_list()
date_index_list=[]
staion_i=station_indexing(d_test['sampling_point'])

for test_i in d_test['Title']:
    date_index_list.append(human_label01.index(test_i))
index_list = [f"{a}_{b}" for a, b in zip(staion_i, date_index_list)]

DN_dict = {key: value for key, value in DN_dict0.items() if key not in index_list}

all_dict=[]
for key in range(1,len([*DN_dict])):
    example=DN_dict[[*DN_dict][key]]
    all_dict.append(example)
        
pretrain_data_part = np.concatenate(all_dict, axis=0)
all_pretrain_data = np.column_stack((np.full(pretrain_data_part.shape[0], lake_name), pretrain_data_part))  
    
pretrain_data0=all_pretrain_data[np.isin(all_pretrain_data[:, 0], subset_lakes), 1:]
pretrain_data0=pretrain_data0.astype(np.float64)

p90=np.percentile(pretrain_data0, 100, axis=0)
mask = np.all(pretrain_data0 <= p90, axis=1)
pretrain_data0=pretrain_data0[mask]
pretrain_data=pretrain_data0

#caculate spectra indiches
band_2 = pretrain_data[:, 0]
band_3 = pretrain_data[:, 1]
band_4 = pretrain_data[:, 2]
band_5 = pretrain_data[:, 3]
band_6 = pretrain_data[:, 4]
band_7 = pretrain_data[:, 5]
band_8 = pretrain_data[:, 6]
band_8A = pretrain_data[:,7]
NDVI1=band_8-band_4
NDVI2=band_8+band_4
NDWI1=band_3-band_8
NDWI2=band_3+band_8

with np.errstate(divide='ignore', invalid='ignore'):
    BR_564=band_5-(band_6+band_4)/2
    
pretrain_dataset = np.column_stack((band_2,band_3,band_4,band_5,band_6,band_7, band_8,band_8A))
pretrain_tensor=torch.Tensor(np.array(pretrain_dataset))
pretrain_tensor=pretrain_tensor.float()
normalized_pretrain, scale_factors, min_vals = minMax(pretrain_tensor)
psuedo_label = np.column_stack(BR_564).T
all_pretrain_pseudo_tensor=torch.Tensor(np.array(psuedo_label))

# unlabeled data
psuedo_label_tensor=all_pretrain_pseudo_tensor
psuedo_label_tensor=psuedo_label_tensor.float()

pretrain_tensor_dataset = TensorDataset(normalized_pretrain, psuedo_label_tensor)
bsize=64
pretrain_tarin_all_loader = DataLoader(pretrain_tensor_dataset, batch_size=bsize, shuffle=True)

# set hyper parameter
pretext_lr = 0.002
downstream_lr = 0.002
lr_set=[pretext_lr,downstream_lr]
do = 0.2
p_weight = 0.1
init_out_ch1 = 48


#### pretext task train part ####

torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

loss_ratio=p_weight
in_ch = 8
out_ch1 = init_out_ch1
out_ch2 = int(out_ch1*1)
out_ch3 = int(out_ch2*0.5)
out_ch4 = int(out_ch3*1)
out_ch5 = int(out_ch4*0.5)
out_ch6 = len(psedo_label_set)
do = do

lr = lr_set[0]
wd = 0
epch = 100
#intv = 40

CNNAE_model = CNNAE().to(device)
optimizer = torch.optim.Adam(CNNAE_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
mse_loss = nn.MSELoss()
exp_lr_scheduler=lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, threshold=1e-4, threshold_mode='rel', cooldown=0, min_lr=0, eps=1e-8, verbose=False)

for epoch in range(epch + 1):
    cnnae_loss=train_AE(CNNAE_model,pretrain_tarin_all_loader,optimizer,mse_loss,scale_factors, min_vals,fit_data_scale_y, fit_data_min_y)
    exp_lr_scheduler.step(cnnae_loss)
    if epoch==100:
        print('Epoch: {}/{}\tLoss: {}'.format(epoch, epch,cnnae_loss))
    
#### downstream task train part ####
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch_m = out_ch5
out_ch2 = out_ch2
out_ch3 = out_ch4
do = do
lr = lr_set[1]
wd = 0
epch = 1000

for parm in CNNAE_model.parameters():
    parm.requires_grad = True 

MLP_model = MLP_base(CNNAE_model).to(device)
MLP_optimizer = torch.optim.Adam(MLP_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
MLP_mse_loss = nn.MSELoss()
MLP_exp_lr_scheduler = lr_scheduler.StepLR(MLP_optimizer, step_size=7, gamma=0.1)        

for epoch in range(epch + 1):
    mlp_loss, train_pred, train_label=train_MLP(MLP_model,train_all_loader,MLP_optimizer,MLP_mse_loss, fit_data_scale_y, fit_data_min_y) #train은 잘 확인할것
    MLP_exp_lr_scheduler.step()
    
    if epoch==1000:
        print('Epoch: {}/{}\tLoss: {}'.format(epoch, epch,mlp_loss))

count0+=1
MLP_model.cpu()
torch.save(MLP_model.state_dict(),'Data\\model_parameter_'+lake_name+'.pt') 
MLP_model.to(device)
y_test_tensor = torch.Tensor(np.array(test_y)).to(device)
x_test_tensor = torch.Tensor(np.array(test_x)).to(device)
x_test_tensor = x_test_tensor.transpose(0, 1).contiguous()
x_test_tensor=torch.unsqueeze(x_test_tensor,0).contiguous()

MLP_model.eval()
pred_8_SL_unlabled = MLP_model(x_test_tensor)

pred_8_SL_unlabled = convert_to_original(pred_8_SL_unlabled, fit_data_scale_y, fit_data_min_y)
y_test_tensor = convert_to_original(y_test_tensor, fit_data_scale_y, fit_data_min_y)  


preformace_tr_x = y_test_tensor.detach().cpu().numpy().reshape(-1)
preformace_tr_y = pred_8_SL_unlabled.detach().cpu().numpy().reshape(-1)

y_train_tensor = torch.Tensor(np.array(train_y)).to(device)
x_train_tensor = torch.Tensor(np.array(train_x)).to(device)
x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()

MLP_model.eval()
pred_train_unlabled = MLP_model(x_train_tensor)

pred_train_unlabled = convert_to_original(pred_train_unlabled, fit_data_scale_y, fit_data_min_y)
y_train_tensor = convert_to_original(y_train_tensor, fit_data_scale_y, fit_data_min_y)  

preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)


#.reshape(-1)
# 1 to 1 plot
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

### plotting
size = 96
y_obs_test_np = y_test_tensor.detach().cpu().numpy()
y_obs_train_np = y_train_tensor.detach().cpu().numpy()

pred_test_np = pred_8_SL_unlabled.detach().cpu().numpy()
pred_train_np = pred_train_unlabled.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

ax.scatter(y_obs_train_np[:, 0], pred_train_np[:, 0], color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
ax.scatter(y_obs_test_np[:, 0], pred_test_np[:, 0], color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
ax.set_xlabel('Measured Chl-a (mg/m3)')
ax.locator_params(axis = "x", nbins = 9)
ax.set_ylabel('Estimated Chl-a (mg/m3)')
ax.xaxis.labelpad = 10
ax.yaxis.labelpad = 10
ax.set_xlim([-1, 100])
ax.set_ylim([-1, 100])
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.close()
plt.clf()

MLP_model.cpu()

####  comparision part SSL-L model  ####
#### pretext task train part ####

torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch = 8
out_ch1 = init_out_ch1
out_ch2 = int(out_ch1)
out_ch3 = int(out_ch2*0.5)
out_ch4 = int(out_ch3)
out_ch5 = int(out_ch4*0.5)
out_ch6 = len(psedo_label_set)
do = do

lr = lr_set[0]
wd = 0
epch = 100

CNNAE_model = CNNAE().to(device)
optimizer = torch.optim.Adam(CNNAE_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
mse_loss = nn.MSELoss()
exp_lr_scheduler=lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, threshold=1e-4, threshold_mode='rel', cooldown=0, min_lr=0, eps=1e-8, verbose=False)

for epoch in range(epch + 1):
    cnnae_loss=train_AE_com(CNNAE_model,train_P_loader,optimizer,mse_loss, fit_data_scale, fit_data_min,fit_data_scale_y, fit_data_min_y)
    exp_lr_scheduler.step(cnnae_loss)
    if epoch==99:
        print('Epoch: {}/{}\tLoss: {}'.format(epoch, epch,cnnae_loss)) 
    
#### downstream task train part ####
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch_m = out_ch5
out_ch2 = out_ch2
out_ch3 = out_ch4
do = do
lr = lr_set[1]
wd = 0
epch = 1000

for parm in CNNAE_model.parameters():
    parm.requires_grad = True 
MLP_model = MLP_base(CNNAE_model).to(device)
MLP_optimizer = torch.optim.Adam(MLP_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
MLP_mse_loss = nn.MSELoss()
MLP_exp_lr_scheduler = lr_scheduler.StepLR(MLP_optimizer, step_size=7, gamma=0.1)        

for epoch in range(epch + 1):
    mlp_loss, train_pred, train_label=train_MLP(MLP_model,train_all_loader,MLP_optimizer,MLP_mse_loss, fit_data_scale_y, fit_data_min_y) #train은 잘 확인할것
    MLP_exp_lr_scheduler.step()
    
MLP_model.cpu()
torch.save(MLP_model.state_dict(), 'Data\model_parameter_SSLTL_'+lake_name+'_'+'.pt')
MLP_model.to(device)

y_test_tensor = torch.Tensor(np.array(test_y)).to(device)
x_test_tensor = torch.Tensor(np.array(test_x)).to(device)
x_test_tensor = x_test_tensor.transpose(0, 1).contiguous()
x_test_tensor=torch.unsqueeze(x_test_tensor,0).contiguous()

MLP_model.eval()
pred_8_SL_unlabled = MLP_model(x_test_tensor)

pred_8_SL_unlabled = convert_to_original(pred_8_SL_unlabled, fit_data_scale_y, fit_data_min_y)
y_test_tensor = convert_to_original(y_test_tensor, fit_data_scale_y, fit_data_min_y)  


preformace_tr_x = y_test_tensor.detach().cpu().numpy().reshape(-1)
preformace_tr_y = pred_8_SL_unlabled.detach().cpu().numpy().reshape(-1)
test_performance=nse(preformace_tr_x,preformace_tr_y)

y_train_tensor = torch.Tensor(np.array(train_y)).to(device)
x_train_tensor = torch.Tensor(np.array(train_x)).to(device)
x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()

MLP_model.eval()
pred_train_unlabled = MLP_model(x_train_tensor)

pred_train_unlabled = convert_to_original(pred_train_unlabled, fit_data_scale_y, fit_data_min_y)
y_train_tensor = convert_to_original(y_train_tensor, fit_data_scale_y, fit_data_min_y)  

preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)
train_performance=nse(preformace_train_x,preformace_train_y)

#.reshape(-1)
# 1 to 1 plot
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

### plotting
size = 96
y_obs_test_np = y_test_tensor.detach().cpu().numpy()
y_obs_train_np = y_train_tensor.detach().cpu().numpy()

pred_test_np = pred_8_SL_unlabled.detach().cpu().numpy()
pred_train_np = pred_train_unlabled.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

ax.scatter(y_obs_train_np[:, 0], pred_train_np[:, 0], color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
ax.scatter(y_obs_test_np[:, 0], pred_test_np[:, 0], color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
ax.set_xlabel('Measured Chl-a (mg/m3)')
ax.locator_params(axis = "x", nbins = 9)
ax.set_ylabel('Estimated Chl-a (mg/m3)')
ax.xaxis.labelpad = 10
ax.yaxis.labelpad = 10
ax.set_xlim([-1, 100])
ax.set_ylim([-1, 100])
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.close()
plt.clf()

MLP_model.cpu()

# Fig 6~7 

In [ ]:
psedo_label_set=['BR_564']
for lake_name in ['Daecheong']: #,'Daecheong'Paldang
    if lake_name == 'Paldang':
        p_weight=10
        init_out_ch1=48
        do=0
        lr_set=[0.001,0.001]
    elif lake_name == 'Daecheong':
        p_weight=1
        init_out_ch1=72
        do=0
        lr_set=[0.001,0.002]
    loss_ratio=p_weight
    
    #### labeled data ####
    lake_name2=new_value=convert_lake_name[lake_name]
    df=df0
    df=df[df['lake_x']==lake_name2]
    plot_size=np.max(df.iloc[:, [6]])
    with pd.option_context('mode.use_inf_as_na', True):
        df['BR_564_L']=df['band_5']-(df['band_6']+df['band_4'])/2
    
    d_train, d_test = train_test_split(df, test_size = 0.2, stratify = df['sampling_point'], random_state = 42)
    
    train_ix, train_x_o, train_y_o= d_train['Unnamed: 0'], d_train.iloc[:,list(range(19, 27))], d_train.iloc[:, [6]]
    test_ix, test_x_o, test_y_o= d_test['Unnamed: 0'], d_test.iloc[:,list(range(19, 27)) ], d_test.iloc[:, [6]] 
    spectra_names = train_x_o.columns
    pig_names = train_y_o.columns
    
    tr_M = train_x_o.max()
    tr_m = train_x_o.min()

    train_x = (train_x_o - tr_m)/(tr_M - tr_m)
    test_x = (test_x_o - tr_m)/(tr_M - tr_m)

    tr_M_y = train_y_o.max()
    tr_m_y = train_y_o.min()

    train_y = (train_y_o - tr_m_y)/(tr_M_y - tr_m_y)
    test_y = (test_y_o - tr_m_y)/(tr_M_y - tr_m_y)

    fit_data_scale=1.0 / (tr_M - tr_m)
    fit_data_min=tr_m
    fit_data_scale = fit_data_scale.tolist()
    fit_data_min = fit_data_min.tolist()

    fit_data_scale_y=1.0 / (tr_M_y - tr_m_y)
    fit_data_min_y=tr_m_y
    fit_data_scale_y = fit_data_scale_y.tolist()
    fit_data_min_y = fit_data_min_y.tolist()
    
    # SSL downstream task's labeled data
    train_x_tensor = torch.Tensor(np.array(train_x))
    train_x_tensor=train_x_tensor.float()
    train_y_tensor= torch.Tensor(np.array(train_y))
    train_y_tensor=train_y_tensor.float()
    
    bsize=10
    train_all =TensorDataset(train_x_tensor, train_y_tensor)
    train_all_loader = DataLoader(train_all, batch_size=bsize, shuffle=True)
    
    # SSL-TL pretrain's pseudo-labeled data 
    index_psedo=psedo_label_indexing(psedo_label_set)
    train_pseudo = d_train.iloc[:, index_psedo]
    test_pseudo=d_test.iloc[:, index_psedo]
    train_p_tensor= torch.Tensor(np.array(train_pseudo))
    train_p_tensor=train_p_tensor.float()

    bsize=10
    label_pretrain=TensorDataset(train_x_tensor, train_p_tensor)
    train_P_loader = DataLoader(label_pretrain, batch_size=bsize, shuffle=True)
    
    #### unlabeled data ####
    
    with open('Data\total_unlabel_data.pkl','rb') as f:
        mydict=pickle.load(f)
    all_pretrain_data=mydict.copy()
        
    subset_lakes=[lake_name]

    DN_dict0=all_pretrain_data[lake_name]
    human_label = pd.read_excel('Data\\'+lake_name2 +'_cloud.xlsx')
    human_label01=human_label['file_name'][(human_label['manual']==0) | (human_label['manual']==1)].to_list()
    date_index_list=[]
    staion_i=station_indexing(d_test['sampling_point'])
    
    for test_i in d_test['Title']:
        date_index_list.append(human_label01.index(test_i))
    index_list = [f"{a}_{b}" for a, b in zip(staion_i, date_index_list)]

    DN_dict = {key: value for key, value in DN_dict0.items() if key not in index_list}

    all_dict=[]
    for key in range(1,len([*DN_dict])):
        example=DN_dict[[*DN_dict][key]]
        all_dict.append(example)
            
    pretrain_data_part = np.concatenate(all_dict, axis=0)
    all_pretrain_data = np.column_stack((np.full(pretrain_data_part.shape[0], lake_name), pretrain_data_part))  
        
    pretrain_data0=all_pretrain_data[np.isin(all_pretrain_data[:, 0], subset_lakes), 1:]
    pretrain_data0=pretrain_data0.astype(np.float64)

    p90=np.percentile(pretrain_data0, 100, axis=0)
    mask = np.all(pretrain_data0 <= p90, axis=1)
    pretrain_data0=pretrain_data0[mask]
    pretrain_data=pretrain_data0

    #caculate spectra indiches
    band_2 = pretrain_data[:, 0]
    band_3 = pretrain_data[:, 1]
    band_4 = pretrain_data[:, 2]
    band_5 = pretrain_data[:, 3]
    band_6 = pretrain_data[:, 4]
    band_7 = pretrain_data[:, 5]
    band_8 = pretrain_data[:, 6]
    band_8A = pretrain_data[:,7]
    NDVI1=band_8-band_4
    NDVI2=band_8+band_4
    NDWI1=band_3-band_8
    NDWI2=band_3+band_8

    with np.errstate(divide='ignore', invalid='ignore'):
        BR_564=band_5-(band_6+band_4)/2
        
    pretrain_dataset = np.column_stack((band_2,band_3,band_4,band_5,band_6,band_7, band_8,band_8A))
    pretrain_tensor=torch.Tensor(np.array(pretrain_dataset))
    pretrain_tensor=pretrain_tensor.float()
    normalized_pretrain, scale_factors, min_vals = minMax(pretrain_tensor)
    psuedo_label = np.column_stack(BR_564).T
    all_pretrain_pseudo_tensor=torch.Tensor(np.array(psuedo_label))

    # unlabeled data
    psuedo_label_tensor=all_pretrain_pseudo_tensor
    psuedo_label_tensor=psuedo_label_tensor.float()

    pretrain_tensor_dataset = TensorDataset(normalized_pretrain, psuedo_label_tensor)
    bsize=64
    pretrain_tarin_all_loader = DataLoader(pretrain_tensor_dataset, batch_size=bsize, shuffle=True)
    
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    random.seed(42)
    np.random.seed(42)
    
    in_ch = 8
    out_ch1 = init_out_ch1
    out_ch2 = int(out_ch1*1)
    out_ch3 = int(out_ch2*0.5)
    out_ch4 = int(out_ch3*1)
    out_ch5 = int(out_ch4*0.5)
    out_ch6 = len(psedo_label_set)
    do = do

    lr = lr_set[0]

    CNNAE_model = CNNAE().to(device)
        
    #### downstream task train part ####
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    random.seed(42)
    np.random.seed(42)

    in_ch_m = out_ch5
    out_ch2 = out_ch2
    out_ch3 = out_ch4
    do = do
    lr = lr_set[1]

    for parm in CNNAE_model.parameters():
        parm.requires_grad = True 
    MLP_model = MLP_base(CNNAE_model).to(device)
    MLP_model.cpu()
    MLP_model.load_state_dict(torch.load('Data\model_parameter_'+lake_name+'.pt'))
    MLP_model.to(device)
    
    y_test_tensor = torch.Tensor(np.array(test_y)).to(device)
    x_test_tensor = torch.Tensor(np.array(test_x)).to(device)
    x_test_tensor = x_test_tensor.transpose(0, 1).contiguous()
    x_test_tensor=torch.unsqueeze(x_test_tensor,0).contiguous()

    MLP_model.eval()
    pred_8_SL_unlabled = MLP_model(x_test_tensor)

    pred_8_SL_unlabled = convert_to_original(pred_8_SL_unlabled, fit_data_scale_y, fit_data_min_y)
    y_test_tensor = convert_to_original(y_test_tensor, fit_data_scale_y, fit_data_min_y)  


    preformace_tr_x = y_test_tensor.detach().cpu().numpy().reshape(-1)
    preformace_tr_y = pred_8_SL_unlabled.detach().cpu().numpy().reshape(-1)

    y_train_tensor = torch.Tensor(np.array(train_y)).to(device)
    x_train_tensor = torch.Tensor(np.array(train_x)).to(device)
    x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
    x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()

    MLP_model.eval()
    pred_train_unlabled = MLP_model(x_train_tensor)

    pred_train_unlabled = convert_to_original(pred_train_unlabled, fit_data_scale_y, fit_data_min_y)
    y_train_tensor = convert_to_original(y_train_tensor, fit_data_scale_y, fit_data_min_y)  

    preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
    preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)
   
    train_nse=nse(preformace_train_x,preformace_train_y)
    test_nse=nse(preformace_tr_x,preformace_tr_y)
    train_smape=smape(preformace_train_x,preformace_train_y)
    test_smape=smape(preformace_tr_x,preformace_tr_y)
    train_rmse=rmse(preformace_train_x,preformace_train_y)
    test_rmse=rmse(preformace_tr_x,preformace_tr_y)
    train_mae=mae(preformace_train_x,preformace_train_y)
    test_mae=mae(preformace_tr_x,preformace_tr_y)
    
    model_performance_dict=pd.DataFrame(columns=['dataset','nse','SMAPE','RMSE','MAE','model','psedo','weight',
                                                 'init_out_ch1', 'do','count0','lr'])
    
    model_performance_dict2=pd.DataFrame({'dataset': 'train', 'nse' : train_nse, 'SMAPE' : train_smape, 'RMSE' : train_rmse, 'MAE' : train_mae,
                                        'model' : 'SSL', 'psedo' : psedo_label_set, 'weight':loss_ratio,
                                        'init_out_ch1' : init_out_ch1, 'do' :do,'count0':count0, 'lr':[lr_set]})
    model_performance_dict=pd.concat([model_performance_dict, model_performance_dict2])
    
    model_performance_dict2=pd.DataFrame({'dataset': 'test','nse' : test_nse,'SMAPE' : test_smape, 'RMSE' : test_rmse, 'MAE' : test_mae,
                                          'model' : 'SSL', 'psedo' : psedo_label_set, 'weight':loss_ratio,
                                        'init_out_ch1' : init_out_ch1, 'do' :do,'count0':count0, 'lr':[lr_set]})
    
    model_performance_dict=pd.concat([model_performance_dict, model_performance_dict2])
    
    #.reshape(-1)
    # 1 to 1 plot
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
    plt.rcParams['font.size'] = 20
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
    plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

    ### plotting
    size = 96
    y_obs_test_np = y_test_tensor.detach().cpu().numpy()
    y_obs_train_np = y_train_tensor.detach().cpu().numpy()

    pred_test_np = pred_8_SL_unlabled.detach().cpu().numpy()
    pred_train_np = pred_train_unlabled.detach().cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 6))
    plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
    plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
    plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

    ax.scatter(y_obs_train_np[:, 0], pred_train_np[:, 0], color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
    ax.scatter(y_obs_test_np[:, 0], pred_test_np[:, 0], color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
    ax.set_xlabel('Measured Chl-a (mg/m3)')
    ax.locator_params(axis = "x", nbins = 9)
    ax.set_ylabel('Estimated Chl-a (mg/m3)')
    ax.xaxis.labelpad = 10
    ax.yaxis.labelpad = 10
    ax.set_xlim([-1, 100])
    ax.set_ylim([-1, 100])
    ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.close()
    plt.clf()
    
    MLP_model.cpu()

    #### comparision part ####
    #### pretext task train part ####

    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    random.seed(42)
    np.random.seed(42)

    in_ch = 8
    out_ch1 = init_out_ch1
    out_ch2 = int(out_ch1)
    out_ch3 = int(out_ch2*0.5)
    out_ch4 = int(out_ch3)
    out_ch5 = int(out_ch4*0.5)
    out_ch6 = len(psedo_label_set)
    do = do

    CNNAE_model = CNNAE().to(device)
                                
    #### downstream task train part ####
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    random.seed(42)
    np.random.seed(42)

    in_ch_m = out_ch5
    out_ch2 = out_ch2
    out_ch3 = out_ch4
    do = do

    for parm in CNNAE_model.parameters():
        parm.requires_grad = True 

    MLP_model = MLP_base(CNNAE_model).to(device)
    MLP_model.cpu()
    MLP_model.load_state_dict(torch.load('Data\\'+'model_parameter_'+lake_name+'.pt'))
    MLP_model.to(device)
    
    y_test_tensor = torch.Tensor(np.array(test_y)).to(device)
    x_test_tensor = torch.Tensor(np.array(test_x)).to(device)
    x_test_tensor = x_test_tensor.transpose(0, 1).contiguous()
    x_test_tensor=torch.unsqueeze(x_test_tensor,0).contiguous()

    MLP_model.eval()
    pred_8_SL_unlabled = MLP_model(x_test_tensor)

    pred_8_SL_unlabled = convert_to_original(pred_8_SL_unlabled, fit_data_scale_y, fit_data_min_y)
    y_test_tensor = convert_to_original(y_test_tensor, fit_data_scale_y, fit_data_min_y)  


    preformace_tr_x = y_test_tensor.detach().cpu().numpy().reshape(-1)
    preformace_tr_y = pred_8_SL_unlabled.detach().cpu().numpy().reshape(-1)

    y_train_tensor = torch.Tensor(np.array(train_y)).to(device)
    x_train_tensor = torch.Tensor(np.array(train_x)).to(device)
    x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
    x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()

    MLP_model.eval()
    pred_train_unlabled = MLP_model(x_train_tensor)

    pred_train_unlabled = convert_to_original(pred_train_unlabled, fit_data_scale_y, fit_data_min_y)
    y_train_tensor = convert_to_original(y_train_tensor, fit_data_scale_y, fit_data_min_y)  

    preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
    preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)
    
    train_nse=nse(preformace_train_x,preformace_train_y)
    test_nse=nse(preformace_tr_x,preformace_tr_y)
    train_smape=smape(preformace_train_x,preformace_train_y)
    test_smape=smape(preformace_tr_x,preformace_tr_y)
    train_rmse=rmse(preformace_train_x,preformace_train_y)
    test_rmse=rmse(preformace_tr_x,preformace_tr_y)
    train_mae=mae(preformace_train_x,preformace_train_y)
    test_mae=mae(preformace_tr_x,preformace_tr_y)
    
    model_performance_dict2=pd.DataFrame({'dataset': 'train', 'nse' : train_nse, 'SMAPE' : train_smape, 'RMSE' : train_rmse, 'MAE' : train_mae,
                                          'model' : 'SSL-L', 'psedo' : psedo_label_set, 'weight':loss_ratio,
                                        'init_out_ch1' : init_out_ch1, 'do' :do,'count0':count0, 'lr':[lr_set]})
    model_performance_dict=pd.concat([model_performance_dict, model_performance_dict2])
    
    model_performance_dict2=pd.DataFrame({'dataset': 'test','nse' : test_nse,'SMAPE' : test_smape, 'RMSE' : test_rmse, 'MAE' : test_mae,
                                          'model' : 'SSL-L', 'psedo' : psedo_label_set, 'weight':loss_ratio,
                                        'init_out_ch1' : init_out_ch1, 'do' :do,'count0':count0, 'lr':[lr_set]})
    model_performance_dict=pd.concat([model_performance_dict, model_performance_dict2])
    
    #.reshape(-1)
    # 1 to 1 plot
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
    plt.rcParams['font.size'] = 20
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
    plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

    ### plotting
    size = 96
    y_obs_test_np = y_test_tensor.detach().cpu().numpy()
    y_obs_train_np = y_train_tensor.detach().cpu().numpy()

    pred_test_np = pred_8_SL_unlabled.detach().cpu().numpy()
    pred_train_np = pred_train_unlabled.detach().cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 6))
    plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
    plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
    plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

    ax.scatter(y_obs_train_np[:, 0], pred_train_np[:, 0], color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
    ax.scatter(y_obs_test_np[:, 0], pred_test_np[:, 0], color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
    ax.set_xlabel('Measured Chl-a (mg/m3)')
    ax.locator_params(axis = "x", nbins = 9)
    ax.set_ylabel('Estimated Chl-a (mg/m3)')
    ax.xaxis.labelpad = 10
    ax.yaxis.labelpad = 10
    ax.set_xlim([-1, 100])
    ax.set_ylim([-1, 100])
    ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.close()
    plt.clf()
    
    MLP_model.cpu()
    model_performance_dict.to_csv('Result\\integrated SSL-DL_SSL-TL'+lake_name+'_performance.csv', encoding='utf-8-sig',index=False)


## Traning & evaulation comparsion models

### Three band model

In [ ]:
model_performance_dict=pd.DataFrame(columns=['dataset','bandratio','nse','SMAPE','RMSE','MAE','lake_name'])
random.seed(42)
np.random.seed(42)

linear_train_x=np.array(train_pseudo['BR_564_L']).reshape(-1, 1)
linear_test_x=np.array(test_pseudo['BR_564_L']).reshape(-1, 1)
linear_train_y=np.array(train_y_o).reshape(-1, 1)
linear_test_y=np.array(test_y_o).reshape(-1, 1)

linear_model=LinearRegression()
linear_model_fit=linear_model.fit(X=linear_train_x, y=linear_train_y)

predict_train_y=linear_model_fit.predict(linear_train_x)
prdict_test_y=linear_model_fit.predict(linear_test_x)

train_nse=nse(linear_train_y,predict_train_y)
test_nse=nse(linear_test_y,prdict_test_y)
train_smape=smape(linear_train_y,predict_train_y)
test_smape=smape(linear_test_y,prdict_test_y)
train_rmse=rmse(linear_train_y,predict_train_y)
test_rmse=rmse(linear_test_y,prdict_test_y)
train_mae=mae(linear_train_y,predict_train_y)
test_mae=mae(linear_test_y,prdict_test_y)

model_performance_dict=model_performance_dict.append({'dataset' : 'train','bandratio' : 'BR_564_L','nse' : train_nse,
                                        'SMAPE' : train_smape, 'RMSE' : train_rmse, 'MAE' : train_mae, 'lake_name' : lake_name}, ignore_index = True)

model_performance_dict=model_performance_dict.append({'dataset' : 'test','bandratio' : 'BR_564_L','nse' : test_nse,
                                        'SMAPE' : test_smape, 'RMSE' : test_rmse, 'MAE' : test_mae, 'lake_name' : lake_name}, ignore_index = True)
model_performance_dict.to_csv('Result\\Threeband_model_'+lake_name+'_performance.csv', encoding='utf-8-sig',index=False)

# 1 to 1 plot
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

### plotting
size = 96

fig, ax = plt.subplots(figsize=(7, 6))
plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

ax.scatter(linear_train_y, predict_train_y, color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
ax.scatter(linear_test_y, prdict_test_y, color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
ax.set_xlabel('Measured Chl-a (mg/m3)')
ax.locator_params(axis = "x", nbins = 9)
ax.set_ylabel('Estimated Chl-a (mg/m3)')
ax.xaxis.labelpad = 10
ax.yaxis.labelpad = 10
ax.set_xlim([-1, 100])
ax.set_ylim([-1, 100])
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# plt.title("Chlorophyll-a", loc = 'left', pad = 30)
plt.tight_layout()
plt.savefig('C:\\Users\\dkskr\\jbs\\Results\\Results_816\\'+ lake_name + "\Band_ratio_" +lake_name+ '_'+str(count0)+'2.png', dpi = 150, bbox_inches = 'tight', pad_inches = 0.1)
plt.close()
plt.clf()

### 1-D CNN model

In [ ]:
model_performance_dict=pd.DataFrame(columns=['dataset','nse','SMAPE','RMSE','MAE','model','psedo','weight'
                                                 'init_out_ch1', 'do','count0','lr'])


torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch = 8
out_ch1 = init_out_ch1
out_ch2 = int(out_ch1)
out_ch3 = int(out_ch2*0.5)
out_ch4 = int(out_ch3)
out_ch5 = int(out_ch4*0.5)
out_ch6 = len(psedo_label_set)
do = do

CNN_part = CNNAE().to(device)

torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch_m = out_ch5
out_ch2 = out_ch2
out_ch3 = out_ch4
do = do
lr = lr_set[1]
wd = 0
epch = 1000

for parm in CNN_part.parameters():
    parm.requires_grad = True 

oneD_CNN = MLP_base(CNN_part).to(device)
oneD_CNN_optimizer = torch.optim.Adam(oneD_CNN.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
oneD_CNN_mse_loss = nn.MSELoss()
oneD_CNN_exp_lr_scheduler = lr_scheduler.StepLR(oneD_CNN_optimizer, step_size=7, gamma=0.1)        

for epoch in range(epch + 1):
    oneD_CNN_loss, train_pred, train_label=train_MLP(oneD_CNN,train_all_loader,oneD_CNN_optimizer,oneD_CNN_mse_loss, fit_data_scale_y, fit_data_min_y) #train은 잘 확인할것
    oneD_CNN_exp_lr_scheduler.step()


y_test_tensor = torch.Tensor(np.array(test_y)).to(device)
x_test_tensor = torch.Tensor(np.array(test_x)).to(device)
x_test_tensor = x_test_tensor.transpose(0, 1).contiguous()
x_test_tensor=torch.unsqueeze(x_test_tensor,0).contiguous()

oneD_CNN.eval()
pred_8_SL_unlabled = oneD_CNN(x_test_tensor)

pred_8_SL_unlabled = convert_to_original(pred_8_SL_unlabled, fit_data_scale_y, fit_data_min_y)
y_test_tensor = convert_to_original(y_test_tensor, fit_data_scale_y, fit_data_min_y)  


preformace_tr_x = y_test_tensor.detach().cpu().numpy().reshape(-1)
preformace_tr_y = pred_8_SL_unlabled.detach().cpu().numpy().reshape(-1)
test_performance=nse(preformace_tr_x,preformace_tr_y)

y_train_tensor = torch.Tensor(np.array(train_y)).to(device)
x_train_tensor = torch.Tensor(np.array(train_x)).to(device)
x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()

oneD_CNN.eval()
pred_train_unlabled = oneD_CNN(x_train_tensor)

pred_train_unlabled = convert_to_original(pred_train_unlabled, fit_data_scale_y, fit_data_min_y)
y_train_tensor = convert_to_original(y_train_tensor, fit_data_scale_y, fit_data_min_y)  

preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)
train_performance=nse(preformace_train_x,preformace_train_y)

train_nse=nse(preformace_train_x,preformace_train_y)
test_nse=nse(preformace_tr_x,preformace_tr_y)
train_smape=smape(preformace_train_x,preformace_train_y)
test_smape=smape(preformace_tr_x,preformace_tr_y)
train_rmse=rmse(preformace_train_x,preformace_train_y)
test_rmse=rmse(preformace_tr_x,preformace_tr_y)
train_mae=mae(preformace_train_x,preformace_train_y)
test_mae=mae(preformace_tr_x,preformace_tr_y)

# 1 to 1 plot
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

### plotting
size = 96
y_obs_test_np = y_test_tensor.detach().cpu().numpy()
y_obs_train_np = y_train_tensor.detach().cpu().numpy()

pred_test_np = pred_8_SL_unlabled.detach().cpu().numpy()
pred_train_np = pred_train_unlabled.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

ax.scatter(y_obs_train_np[:, 0], pred_train_np[:, 0], color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
ax.scatter(y_obs_test_np[:, 0], pred_test_np[:, 0], color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
ax.set_xlabel('Measured Chl-a (mg/m3)')
ax.locator_params(axis = "x", nbins = 9)
ax.set_ylabel('Estimated Chl-a (mg/m3)')
ax.xaxis.labelpad = 10
ax.yaxis.labelpad = 10
ax.set_xlim([-1, 100])
ax.set_ylim([-1, 100])
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.close()
plt.clf()

oneD_CNN.cpu()
model_performance_dict.to_csv('Result\\1DCNN_'+lake_name+'_performance.csv', encoding='utf-8-sig',index=False)


## Generative SSL

In [ ]:
psedo_label_set=['BR_564']
for lake_name in ['Daecheong']: #,'Daecheong'Paldang
    if lake_name == 'Paldang':
        p_weight=10
        init_out_ch1=48
        do=0
        lr_set=[0.001,0.001]
        count0=271
    elif lake_name == 'Daecheong':
        p_weight=1
        init_out_ch1=72
        do=0
        lr_set=[0.001,0.002]
        count0=218
    loss_ratio=p_weight
    
    #### labeled data ####
    lake_name2=new_value=convert_lake_name[lake_name]
    df=df0
    df=df[df['lake_x']==lake_name2]
    plot_size=np.max(df.iloc[:, [6]])
    with pd.option_context('mode.use_inf_as_na', True):
        df['BR_564_L']=df['band_5']-(df['band_6']+df['band_4'])/2
    
    d_train, d_test = train_test_split(df, test_size = 0.2, stratify = df['sampling_point'], random_state = 42)
    
    train_ix, train_x_o, train_y_o= d_train['Unnamed: 0'], d_train.iloc[:,list(range(19, 27))], d_train.iloc[:, [6]]
    test_ix, test_x_o, test_y_o= d_test['Unnamed: 0'], d_test.iloc[:,list(range(19, 27)) ], d_test.iloc[:, [6]] 
    spectra_names = train_x_o.columns
    pig_names = train_y_o.columns
    
    tr_M = train_x_o.max()
    tr_m = train_x_o.min()

    train_x = (train_x_o - tr_m)/(tr_M - tr_m)
    test_x = (test_x_o - tr_m)/(tr_M - tr_m)

    tr_M_y = train_y_o.max()
    tr_m_y = train_y_o.min()

    train_y = (train_y_o - tr_m_y)/(tr_M_y - tr_m_y)
    test_y = (test_y_o - tr_m_y)/(tr_M_y - tr_m_y)

    fit_data_scale=1.0 / (tr_M - tr_m)
    fit_data_min=tr_m
    fit_data_scale = fit_data_scale.tolist()
    fit_data_min = fit_data_min.tolist()

    fit_data_scale_y=1.0 / (tr_M_y - tr_m_y)
    fit_data_min_y=tr_m_y
    fit_data_scale_y = fit_data_scale_y.tolist()
    fit_data_min_y = fit_data_min_y.tolist()
    
    # SSL downstream task's labeled data
    train_x_tensor = torch.Tensor(np.array(train_x))
    train_x_tensor=train_x_tensor.float()
    train_y_tensor= torch.Tensor(np.array(train_y))
    train_y_tensor=train_y_tensor.float()
    
    bsize=10
    train_all =TensorDataset(train_x_tensor, train_y_tensor)
    train_all_loader = DataLoader(train_all, batch_size=bsize, shuffle=True)
    
    # SSL-L pretrain's pseudo-labeled data 
    index_psedo=psedo_label_indexing(psedo_label_set)
    train_pseudo = d_train.iloc[:, index_psedo]
    test_pseudo=d_test.iloc[:, index_psedo]
    train_p_tensor= torch.Tensor(np.array(train_pseudo))
    train_p_tensor=train_p_tensor.float()

    bsize=10
    label_pretrain=TensorDataset(train_x_tensor, train_p_tensor)
    train_P_loader = DataLoader(label_pretrain, batch_size=bsize, shuffle=True)
    
    #### unlabeled data ####
    
    with open('Data\\total_unlabel_data.pkl','rb') as f:
        mydict=pickle.load(f)
    all_pretrain_data=mydict.copy()
        
    subset_lakes=[lake_name]

    DN_dict0=all_pretrain_data[lake_name]
    human_label = pd.read_excel('Data\\'+lake_name2 +'_cloud.xlsx')
    human_label01=human_label['file_name'][(human_label['manual']==0) | (human_label['manual']==1)].to_list()
    date_index_list=[]
    staion_i=station_indexing(d_test['sampling_point'])
    
    for test_i in d_test['Title']:
        date_index_list.append(human_label01.index(test_i))
    index_list = [f"{a}_{b}" for a, b in zip(staion_i, date_index_list)]

    DN_dict = {key: value for key, value in DN_dict0.items() if key not in index_list}

    all_dict=[]
    for key in range(1,len([*DN_dict])):
        example=DN_dict[[*DN_dict][key]]
        all_dict.append(example)
            
    pretrain_data_part = np.concatenate(all_dict, axis=0)
    all_pretrain_data = np.column_stack((np.full(pretrain_data_part.shape[0], lake_name), pretrain_data_part))  
        
    pretrain_data0=all_pretrain_data[np.isin(all_pretrain_data[:, 0], subset_lakes), 1:]
    pretrain_data0=pretrain_data0.astype(np.float64)

    p90=np.percentile(pretrain_data0, 100, axis=0)
    mask = np.all(pretrain_data0 <= p90, axis=1)
    pretrain_data0=pretrain_data0[mask]
    pretrain_data=pretrain_data0

    #caculate spectra indiches
    band_2 = pretrain_data[:, 0]
    band_3 = pretrain_data[:, 1]
    band_4 = pretrain_data[:, 2]
    band_5 = pretrain_data[:, 3]
    band_6 = pretrain_data[:, 4]
    band_7 = pretrain_data[:, 5]
    band_8 = pretrain_data[:, 6]
    band_8A = pretrain_data[:,7]
    NDVI1=band_8-band_4
    NDVI2=band_8+band_4
    NDWI1=band_3-band_8
    NDWI2=band_3+band_8

    with np.errstate(divide='ignore', invalid='ignore'):
        BR_564=band_5-(band_6+band_4)/2
        
    pretrain_dataset = np.column_stack((band_2,band_3,band_4,band_5,band_6,band_7, band_8,band_8A))
    pretrain_tensor=torch.Tensor(np.array(pretrain_dataset))
    pretrain_tensor=pretrain_tensor.float()
    normalized_pretrain, scale_factors, min_vals = minMax(pretrain_tensor)
    psuedo_label = np.column_stack(BR_564).T
    all_pretrain_pseudo_tensor=torch.Tensor(np.array(psuedo_label))

    # unlabeled data
    psuedo_label_tensor=all_pretrain_pseudo_tensor
    psuedo_label_tensor=psuedo_label_tensor.float()

    pretrain_tensor_dataset = TensorDataset(normalized_pretrain, psuedo_label_tensor)
    bsize=64
    pretrain_tarin_all_loader = DataLoader(pretrain_tensor_dataset, batch_size=bsize, shuffle=True)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

loss_ratio=p_weight
in_ch = 8
out_ch1 = init_out_ch1
out_ch2 = int(out_ch1*1)
out_ch3 = int(out_ch2*0.5)
out_ch4 = int(out_ch3*1)
out_ch5 = int(out_ch4*0.5)
out_ch6 = len(psedo_label_set)
do = do

lr = lr_set[0]
wd = 0
epch = 100

CNNAE_no_WSL_model = CNNAE_no_WSL().to(device)
optimizer = torch.optim.Adam(CNNAE_no_WSL_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
mse_loss = nn.MSELoss()
exp_lr_scheduler=lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, threshold=1e-4, threshold_mode='rel', cooldown=0, min_lr=0, eps=1e-8, verbose=False)

for epoch in range(epch + 1):
    cnnae_loss=train_AE_no_wsl(CNNAE_no_WSL_model,pretrain_tarin_all_loader,optimizer,mse_loss,scale_factors, min_vals,fit_data_scale_y, fit_data_min_y)
    exp_lr_scheduler.step(cnnae_loss)
    if epoch==100:
        print('Epoch: {}/{}\tLoss: {}'.format(epoch, epch,cnnae_loss))

#### downstream task train part ####
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch_m = out_ch5
out_ch2 = out_ch2
out_ch3 = out_ch4
do = do
lr = lr_set[1]
wd = 0
epch = 1000

for parm in CNNAE_no_WSL_model.parameters():
    parm.requires_grad = True 

MLP_model = MLP_base(CNNAE_no_WSL_model).to(device)
MLP_optimizer = torch.optim.Adam(MLP_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
MLP_mse_loss = nn.MSELoss()
MLP_exp_lr_scheduler = lr_scheduler.StepLR(MLP_optimizer, step_size=7, gamma=0.1)        

for epoch in range(epch + 1):
    mlp_loss, train_pred, train_label=train_MLP(MLP_model,train_all_loader,MLP_optimizer,MLP_mse_loss, fit_data_scale_y, fit_data_min_y) #train은 잘 확인할것
    MLP_exp_lr_scheduler.step()

    if epoch==1000:
        print('Epoch: {}/{}\tLoss: {}'.format(epoch, epch,mlp_loss))

count0+=1

y_test_tensor = torch.Tensor(np.array(test_y)).to(device)
x_test_tensor = torch.Tensor(np.array(test_x)).to(device)
x_test_tensor = x_test_tensor.transpose(0, 1).contiguous()
x_test_tensor=torch.unsqueeze(x_test_tensor,0).contiguous()

MLP_model.eval()
pred_8_SL_unlabled = MLP_model(x_test_tensor)
pred_8_SL_unlabled = convert_to_original(pred_8_SL_unlabled, fit_data_scale_y, fit_data_min_y)
y_test_tensor = convert_to_original(y_test_tensor, fit_data_scale_y, fit_data_min_y)  


preformace_tr_x = y_test_tensor.detach().cpu().numpy().reshape(-1)
preformace_tr_y = pred_8_SL_unlabled.detach().cpu().numpy().reshape(-1)

y_train_tensor = torch.Tensor(np.array(train_y)).to(device)
x_train_tensor = torch.Tensor(np.array(train_x)).to(device)
x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()

MLP_model.eval()
pred_train_unlabled = MLP_model(x_train_tensor)

pred_train_unlabled = convert_to_original(pred_train_unlabled, fit_data_scale_y, fit_data_min_y)
y_train_tensor = convert_to_original(y_train_tensor, fit_data_scale_y, fit_data_min_y)  

preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)

preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)
train_nse=nse(preformace_train_x,preformace_train_y)
test_nse=nse(preformace_tr_x,preformace_tr_y)
train_smape=smape(preformace_train_x,preformace_train_y)
test_smape=smape(preformace_tr_x,preformace_tr_y)
train_rmse=rmse(preformace_train_x,preformace_train_y)
test_rmse=rmse(preformace_tr_x,preformace_tr_y)
train_mae=mae(preformace_train_x,preformace_train_y)
test_mae=mae(preformace_tr_x,preformace_tr_y)


#.reshape(-1)
# 1 to 1 plot
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

### plotting
size = 96
y_obs_test_np = y_test_tensor.detach().cpu().numpy()
y_obs_train_np = y_train_tensor.detach().cpu().numpy()

pred_test_np = pred_8_SL_unlabled.detach().cpu().numpy()
pred_train_np = pred_train_unlabled.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

ax.scatter(y_obs_train_np[:, 0], pred_train_np[:, 0], color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
ax.scatter(y_obs_test_np[:, 0], pred_test_np[:, 0], color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
ax.set_xlabel('Measured Chl-a (mg/m3)')
ax.locator_params(axis = "x", nbins = 9)
ax.set_ylabel('Estimated Chl-a (mg/m3)')
ax.xaxis.labelpad = 10
ax.yaxis.labelpad = 10
ax.set_xlim([-1, 100])
ax.set_ylim([-1, 100])
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.close()
plt.clf()

MLP_model.cpu()
model_performance_dict.to_csv('Result\\'+'Generative_SSL-DL_'+lake_name+'_performance.csv', encoding='utf-8-sig',index=False)

## Predictive SSL 

In [ ]:
psedo_label_set=['BR_564']
for lake_name in ['Paldang']: #,'Daecheong'Paldang
    if lake_name == 'Paldang':
        p_weight=10
        init_out_ch1=48
        do=0
        lr_set=[0.001,0.001]
    elif lake_name == 'Daecheong':
        p_weight=1
        init_out_ch1=72
        do=0
        lr_set=[0.001,0.002]
    loss_ratio=p_weight
    #### labeled data ####
    lake_name2=new_value=convert_lake_name[lake_name]
    df=df0
    df=df[df['lake_x']==lake_name2]
    plot_size=np.max(df.iloc[:, [6]])
    with pd.option_context('mode.use_inf_as_na', True):
        df['BR_564_L']=df['band_5']-(df['band_6']+df['band_4'])/2
    
    d_train, d_test = train_test_split(df, test_size = 0.2, stratify = df['sampling_point'], random_state = 42)
    
    train_ix, train_x_o, train_y_o= d_train['Unnamed: 0'], d_train.iloc[:,list(range(19, 27))], d_train.iloc[:, [6]]
    test_ix, test_x_o, test_y_o= d_test['Unnamed: 0'], d_test.iloc[:,list(range(19, 27)) ], d_test.iloc[:, [6]] 
    spectra_names = train_x_o.columns
    pig_names = train_y_o.columns
    
    tr_M = train_x_o.max()
    tr_m = train_x_o.min()

    train_x = (train_x_o - tr_m)/(tr_M - tr_m)
    test_x = (test_x_o - tr_m)/(tr_M - tr_m)

    tr_M_y = train_y_o.max()
    tr_m_y = train_y_o.min()

    train_y = (train_y_o - tr_m_y)/(tr_M_y - tr_m_y)
    test_y = (test_y_o - tr_m_y)/(tr_M_y - tr_m_y)

    fit_data_scale=1.0 / (tr_M - tr_m)
    fit_data_min=tr_m
    fit_data_scale = fit_data_scale.tolist()
    fit_data_min = fit_data_min.tolist()

    fit_data_scale_y=1.0 / (tr_M_y - tr_m_y)
    fit_data_min_y=tr_m_y
    fit_data_scale_y = fit_data_scale_y.tolist()
    fit_data_min_y = fit_data_min_y.tolist()
    
    # SSL downstream task's labeled data
    train_x_tensor = torch.Tensor(np.array(train_x))
    train_x_tensor=train_x_tensor.float()
    train_y_tensor= torch.Tensor(np.array(train_y))
    train_y_tensor=train_y_tensor.float()
    
    bsize=10
    train_all =TensorDataset(train_x_tensor, train_y_tensor)
    train_all_loader = DataLoader(train_all, batch_size=bsize, shuffle=True)
    
    # SSL-L pretrain's pseudo-labeled data 
    index_psedo=psedo_label_indexing(psedo_label_set)
    train_pseudo = d_train.iloc[:, index_psedo]
    test_pseudo=d_test.iloc[:, index_psedo]
    train_p_tensor= torch.Tensor(np.array(train_pseudo))
    train_p_tensor=train_p_tensor.float()

    bsize=10
    label_pretrain=TensorDataset(train_x_tensor, train_p_tensor)
    train_P_loader = DataLoader(label_pretrain, batch_size=bsize, shuffle=True)
    
    #### unlabeled data ####
    
    with open('Data\\total_unlabel_data.pkl','rb') as f:
        mydict=pickle.load(f)
    all_pretrain_data=mydict.copy()
        
    subset_lakes=[lake_name]

    DN_dict0=all_pretrain_data[lake_name]
    human_label = pd.read_excel('Data\\'+lake_name2 +'_cloud.xlsx')
    human_label01=human_label['file_name'][(human_label['manual']==0) | (human_label['manual']==1)].to_list()
    date_index_list=[]
    staion_i=station_indexing(d_test['sampling_point'])
    
    for test_i in d_test['Title']:
        date_index_list.append(human_label01.index(test_i))
    index_list = [f"{a}_{b}" for a, b in zip(staion_i, date_index_list)]

    DN_dict = {key: value for key, value in DN_dict0.items() if key not in index_list}

    all_dict=[]
    for key in range(1,len([*DN_dict])):
        example=DN_dict[[*DN_dict][key]]
        all_dict.append(example)
            
    pretrain_data_part = np.concatenate(all_dict, axis=0)
    all_pretrain_data = np.column_stack((np.full(pretrain_data_part.shape[0], lake_name), pretrain_data_part))  
        
    pretrain_data0=all_pretrain_data[np.isin(all_pretrain_data[:, 0], subset_lakes), 1:]
    pretrain_data0=pretrain_data0.astype(np.float64)

    p90=np.percentile(pretrain_data0, 100, axis=0)
    mask = np.all(pretrain_data0 <= p90, axis=1)
    pretrain_data0=pretrain_data0[mask]
    pretrain_data=pretrain_data0

    #caculate spectra indiches
    band_2 = pretrain_data[:, 0]
    band_3 = pretrain_data[:, 1]
    band_4 = pretrain_data[:, 2]
    band_5 = pretrain_data[:, 3]
    band_6 = pretrain_data[:, 4]
    band_7 = pretrain_data[:, 5]
    band_8 = pretrain_data[:, 6]
    band_8A = pretrain_data[:,7]
    NDVI1=band_8-band_4
    NDVI2=band_8+band_4
    NDWI1=band_3-band_8
    NDWI2=band_3+band_8

    with np.errstate(divide='ignore', invalid='ignore'):
        BR_564=band_5-(band_6+band_4)/2
        
    pretrain_dataset = np.column_stack((band_2,band_3,band_4,band_5,band_6,band_7, band_8,band_8A))
    pretrain_tensor=torch.Tensor(np.array(pretrain_dataset))
    pretrain_tensor=pretrain_tensor.float()
    normalized_pretrain, scale_factors, min_vals = minMax(pretrain_tensor)
    psuedo_label = np.column_stack(BR_564).T
    all_pretrain_pseudo_tensor=torch.Tensor(np.array(psuedo_label))

    # unlabeled data
    psuedo_label_tensor=all_pretrain_pseudo_tensor
    psuedo_label_tensor=psuedo_label_tensor.float()

    pretrain_tensor_dataset = TensorDataset(normalized_pretrain, psuedo_label_tensor)
    bsize=64
    pretrain_tarin_all_loader = DataLoader(pretrain_tensor_dataset, batch_size=bsize, shuffle=True)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

loss_ratio=p_weight
in_ch = 8
out_ch1 = init_out_ch1
out_ch2 = int(out_ch1*1)
out_ch3 = int(out_ch2*0.5)
out_ch4 = int(out_ch3*1)
out_ch5 = int(out_ch4*0.5)
out_ch6 = len(psedo_label_set)
do = do

lr = lr_set[0]
wd = 0
epch = 100
#intv = 40

CNNAE_pred_model = Pred_CNNAE().to(device)
optimizer = torch.optim.Adam(CNNAE_pred_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
mse_loss = nn.MSELoss()
exp_lr_scheduler=lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, threshold=1e-4, threshold_mode='rel', cooldown=0, min_lr=0, eps=1e-8, verbose=False)

for epoch in range(epch + 1):
    cnnae_loss=train_AE_pred(CNNAE_pred_model,pretrain_tarin_all_loader,optimizer,mse_loss,scale_factors, min_vals,fit_data_scale_y, fit_data_min_y)
    exp_lr_scheduler.step(cnnae_loss)
    if epoch==100:
        print('Epoch: {}/{}\tLoss: {}'.format(epoch, epch,cnnae_loss)) 

#### downstream task train part ####
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch_m = out_ch5
out_ch2 = out_ch2
out_ch3 = out_ch4
do = do
lr = lr_set[1]
wd = 0
epch = 1000

for parm in CNNAE_pred_model.parameters():
    parm.requires_grad = True 

MLP_model = MLP_base(CNNAE_pred_model).to(device)
MLP_optimizer = torch.optim.Adam(MLP_model.parameters(), lr = lr, eps = 1e-4, weight_decay = wd)
MLP_mse_loss = nn.MSELoss()
MLP_exp_lr_scheduler = lr_scheduler.StepLR(MLP_optimizer, step_size=7, gamma=0.1)        

for epoch in range(epch + 1):
    mlp_loss, train_pred, train_label=train_MLP(MLP_model,train_all_loader,MLP_optimizer,MLP_mse_loss, fit_data_scale_y, fit_data_min_y) #train은 잘 확인할것
    MLP_exp_lr_scheduler.step()

    if epoch==1000:
        print('Epoch: {}/{}\tLoss: {}'.format(epoch, epch,mlp_loss))

count0+=1

y_test_tensor = torch.Tensor(np.array(test_y)).to(device)
x_test_tensor = torch.Tensor(np.array(test_x)).to(device)
x_test_tensor = x_test_tensor.transpose(0, 1).contiguous()
x_test_tensor=torch.unsqueeze(x_test_tensor,0).contiguous()

MLP_model.eval()
pred_8_SL_unlabled = MLP_model(x_test_tensor)
pred_8_SL_unlabled = convert_to_original(pred_8_SL_unlabled, fit_data_scale_y, fit_data_min_y)
y_test_tensor = convert_to_original(y_test_tensor, fit_data_scale_y, fit_data_min_y)  


preformace_tr_x = y_test_tensor.detach().cpu().numpy().reshape(-1)
preformace_tr_y = pred_8_SL_unlabled.detach().cpu().numpy().reshape(-1)

y_train_tensor = torch.Tensor(np.array(train_y)).to(device)
x_train_tensor = torch.Tensor(np.array(train_x)).to(device)
x_train_tensor = x_train_tensor.transpose(0, 1).contiguous()
x_train_tensor=torch.unsqueeze(x_train_tensor,0).contiguous()

MLP_model.eval()
pred_train_unlabled = MLP_model(x_train_tensor)

pred_train_unlabled = convert_to_original(pred_train_unlabled, fit_data_scale_y, fit_data_min_y)
y_train_tensor = convert_to_original(y_train_tensor, fit_data_scale_y, fit_data_min_y)  

preformace_train_x = y_train_tensor.detach().cpu().numpy().reshape(-1)
preformace_train_y = pred_train_unlabled.detach().cpu().numpy().reshape(-1)
train_nse=nse(preformace_train_x,preformace_train_y)
test_nse=nse(preformace_tr_x,preformace_tr_y)
train_smape=smape(preformace_train_x,preformace_train_y)
test_smape=smape(preformace_tr_x,preformace_tr_y)
train_rmse=rmse(preformace_train_x,preformace_train_y)
test_rmse=rmse(preformace_tr_x,preformace_tr_y)
train_mae=mae(preformace_train_x,preformace_train_y)
test_mae=mae(preformace_tr_x,preformace_tr_y)

#.reshape(-1)
# 1 to 1 plot
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

### plotting
size = 96
y_obs_test_np = y_test_tensor.detach().cpu().numpy()
y_obs_train_np = y_train_tensor.detach().cpu().numpy()

pred_test_np = pred_8_SL_unlabled.detach().cpu().numpy()
pred_train_np = pred_train_unlabled.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
plt.grid(True, axis = 'y', color = '#cccc', alpha = 0.9, zorder = -1)
plt.grid(True, axis = 'x', color = '#cccc', alpha = 0.9, zorder = -1)
plt.plot([-1, 1000], [-1, 1000], 'black', linestyle = 'dashed', linewidth = 0.5, zorder=10)

ax.scatter(y_obs_train_np[:, 0], pred_train_np[:, 0], color = 'dodgerblue', marker = "s", alpha = 0.7, s = size, edgecolors = "k", zorder = 5) ## training
ax.scatter(y_obs_test_np[:, 0], pred_test_np[:, 0], color = 'red', marker = "o", alpha = 0.7, s = size, edgecolors = "k", zorder = 7) ### validataion
ax.set_xlabel('Measured Chl-a (mg/m3)')
ax.locator_params(axis = "x", nbins = 9)
ax.set_ylabel('Estimated Chl-a (mg/m3)')
ax.xaxis.labelpad = 10
ax.yaxis.labelpad = 10
ax.set_xlim([-1, 100])
ax.set_ylim([-1, 100])
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# plt.title("Chlorophyll-a", loc = 'left', pad = 30)
plt.tight_layout()
plt.close()
plt.clf()

MLP_model.cpu()
model_performance_dict.to_csv('Result\\'+'Predicted_SSL-DL_'+lake_name+'_performance.csv', encoding='utf-8-sig',index=False)

# Fig 8~9

In [ ]:
lake_name='Paldang'
lake_name2=new_value=convert_lake_name[lake_name]
df=df0
df=df[df['lake_x']==lake_name2]
plot_size=np.max(df.iloc[:, [6]])
count0=0
with pd.option_context('mode.use_inf_as_na', True):
    df['BR_564_L']=df['band_5']-(df['band_6']+df['band_4'])/2

d_train, d_test = train_test_split(df, test_size = 0.2, stratify = df['sampling_point'], random_state = 42)

train_ix, train_x_o, train_y_o= d_train['Unnamed: 0'], d_train.iloc[:,list(range(19, 27))], d_train.iloc[:, [6]]
test_ix, test_x_o, test_y_o= d_test['Unnamed: 0'], d_test.iloc[:,list(range(19, 27)) ], d_test.iloc[:, [6]] 
spectra_names = train_x_o.columns
pig_names = train_y_o.columns

tr_M = train_x_o.max()
tr_m = train_x_o.min()

train_x = (train_x_o - tr_m)/(tr_M - tr_m)
test_x = (test_x_o - tr_m)/(tr_M - tr_m)

tr_M_y = train_y_o.max()
tr_m_y = train_y_o.min()
human_label = pd.read_excel('Data\\'+lake_name2 +'_cloud.xlsx')
human_label01=human_label['file_name'][(human_label['manual']==0) | (human_label['manual']==1)].to_list()
psedo_label_set=['BR_564']

mask = pickle.load(open("Data\\image_masking.pkl", 'rb'))[lake_name2]
mask = np.ma.masked_where(mask == 0, mask)
mask = np.where(np.nan_to_num(np.array(mask), 0) != 0)
    
    
if lake_name == 'Paldang':
    p_weight=10
    init_out_ch1=48
    do=0
    lr_set=[0.001,0.001]
elif lake_name == 'Daecheong':
    p_weight=1
    init_out_ch1=72
    do=0
    lr_set=[0.001,0.002]

torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch = 8
out_ch1 = init_out_ch1
out_ch2 = int(out_ch1*1)
out_ch3 = int(out_ch2*0.5)
out_ch4 = int(out_ch3*1)
out_ch5 = int(out_ch4*0.5)
out_ch6 = len(psedo_label_set)
do = do

lr = lr_set[0]

CNNAE_model = CNNAE().to(device)
    
#### downstream task train part ####
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)

in_ch_m = out_ch5
out_ch2 = out_ch2
out_ch3 = out_ch4
do = do
lr = lr_set[1]

for parm in CNNAE_model.parameters():
    parm.requires_grad = True 
MLP_model = MLP_base(CNNAE_model).to(device)
MLP_model.cpu()

MLP_model.load_state_dict(torch.load('Data\\model_parameter_'+lake_name+'.pt'))
MLP_model.to(device)

In [ ]:
MLP_model.cpu()
#### Read Sentinel-2 image, masking, and background image
#### RETRIEVE MAPPING RESULT

img0 = rasterio.open("Data\\1_11_20210308T021559_20210308T021602_T52SCG.tif")
img=img0.read()

#### RGB BACKGROUND MAP
bg = np.stack((img[3, :, :],
               img[2, :, :],
               img[1, :, :]), axis = 2)

left, bottom, right, top = img0.bounds
est_chla_maps='None'

img_inp =  img[1:9, : , :].transpose(1, 2, 0)[mask]
img_inp_mM = (img_inp - np.array(tr_m))/(np.array(tr_M) - np.array(tr_m))
img_inp_mM = img_inp_mM.transpose(1, 0)

img_inp_mM = torch.Tensor(img_inp_mM).unsqueeze(0)

est_chla_ = MLP_model(img_inp_mM).detach().numpy() #.to(device)

### Re-scaling
est_chla_ = est_chla_ * (np.array(tr_M_y) - np.array(tr_m_y)) + np.array(tr_m_y)

est_chla_maps = np.empty((bg.shape[0], bg.shape[1], 1))
est_chla_maps[:] = np.nan
est_chla_maps[mask] = est_chla_

plt.figure(figsize = (bg.shape[0]/200, bg.shape[1]/200))
ax = plt.gca()

img1 = plt.imshow(bg/4800, extent=[left, right, bottom, top])
img2 = plt.imshow(est_chla_maps,extent=[left, right, bottom, top])
img2.set_clim(0, 50)
plt.jet()

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
plt.colorbar(img2, cax = cax)

x_ticks = np.arange(np.floor(left), np.ceil(right), 2000)
y_ticks = np.arange(np.floor(bottom), np.ceil(top), 2000)

plt.show()


# FIG 4

In [ ]:
import seaborn as sns

target_lakes='Paldang'
df=pd.read_csv('Data\\Figure4_'+target_lakes+'.csv',encoding='utf-8-sig')
df = df[df['sampling_date']>=20190101]

In [ ]:
df['sampling_date'] = pd.to_datetime(df['sampling_date'], format='%Y%m%d')

df['Year'] = df['sampling_date'].dt.year
df['Month'] = df['sampling_date'].dt.month

monthly_avg = df.groupby(['Year', 'Month', 'sampling_point'])['chla'].mean().reset_index()

plt.figure(figsize=(15, 8))
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 20
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'
colors=['#ff6666', '#ff9966', '#ffe566', '#66c266', '#668cff', '#c266c2'] #, '#d699d6'
#'#ff9999', '#ffb899', '#ffed99', '#99d699', '#99c2ff', '#d699d6'

sns.barplot(data=monthly_avg, 
           x='Month',
           y='chla',
           hue='sampling_point',
           ci=None,
           legend=False,
           palette=colors,
           edgecolor='black',
           )
plt.ylim(top=50)

plt.xlabel('Month') 
plt.ylabel('Chl-a (mg/m3)')


plt.xticks(range(12), ['1', '2', '3', '4', '5', '6', 
                      '7', '8', '9', '10', '11', '12'])
plt.axhline(y=df['chla'].mean(), color='red', linestyle='--', lw=3)

plt.grid(True, axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# FIG 5

In [ ]:
df=df0
df=df[df['lake_x']=='Daecheongdam']
df.columns=['Unnamed: 0', 'WD', 'basin', 'sampling_point', 'sampling_date', 'BOD',
       'Chl-a', 'COD', 'DO', 'EC', 'SS', 'wtemp', 'TOC', 'clar', 'lake_x',
       'Title', 'Satellite_date', 'check_same_date', 'B1', 'B2',
       'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8a','B9',
       'B11', 'B12', 'check']
df[['B1', 'B2','B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8a','B9','B11', 'B12']]=df[['B1', 'B2','B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8a','B9','B11', 'B12']]*0.0001
df.columns
with pd.option_context('mode.use_inf_as_na', True):
       df['Three-Band']=df['B5']-(df['B6']+df['B4'])/2

## Figure 5 bd

In [83]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

def plot_reflectance_chla(data, bands, chla_range, chla_column='Chl-a',reflectance_columns=None,
                        figsize=(12, 8), dpi=100, output_path=None):
    
    if reflectance_columns is None:
        raise ValueError("reflectance_columns must be specified")
    
    if len(bands) != len(reflectance_columns):
        raise ValueError("Number of bands must match number of reflectance columns")
    
    # 그래프 생성
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
    plt.rcParams['font.size'] = 20
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
    plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'
    

    #colors = ['darkred', 'red', 'orange', 'yellow']
    colors = ['yellow', 'orange', 'red', 'darkred']
    n_bins = 256
    custom_colormap = LinearSegmentedColormap.from_list('custom_red_yellow', colors, N=n_bins)
    
    x_positions = np.arange(len(bands))
    
    sorted_data = data.sort_values(by=chla_column, ascending=True)

    for idx, row in sorted_data.iterrows():

        chla_min, chla_max=chla_range
        color_value = (row[chla_column] - chla_min) / (chla_max - chla_min)
        color_value = max(0, min(1, color_value))
        color = custom_colormap(color_value)
        
        reflectance_values = row[reflectance_columns]
        ax.plot(x_positions, reflectance_values, color=color, linewidth=1, 
                label=f'chla {row[chla_column]:.2f}')
    
    ax.set_xticks(x_positions)
    ax.set_xticklabels(bands, rotation=0)
    ax.set_ylabel('Reflectance')
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_ylim(0, 0.35)

    norm = plt.Normalize(*chla_range)
    sm = plt.cm.ScalarMappable(cmap=custom_colormap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm)
    cbar.set_label('Chl-a Concentration')

    plt.tight_layout()
    

    if output_path:
        plt.savefig(output_path, bbox_inches='tight')
    else:
        plt.show()
    
    plt.close()

In [ ]:
if __name__ == "__main__":    
    wavelengths=['B1', 'B2','B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8a','B9','B11', 'B12']

    plot_reflectance_chla(df, bands=wavelengths, reflectance_columns = wavelengths, chla_range=(0, 50))

## Figure 5 ac

In [85]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_correlation(df, columns=None, figsize=(14, 12), 
                    cmap='coolwarm', annot=True, fmt='.2f',
                    title='Correlation Matrix',
                    cbar_shrink=0.5,   
                    cbar_aspect=20):     
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
    plt.rcParams['font.size'] = 20
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
    plt.rcParams['mathtext.bf'] = 'Times New Roman:italic:bold'

    if columns is None:

        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df_corr = df[numeric_cols]
    else:
        df_corr = df[columns]

    corr_matrix = df_corr.corr()

    plt.figure(figsize=figsize)
    

    mask = np.triu(np.ones_like(corr_matrix), k=1)  
    sns.heatmap(corr_matrix, 
                mask=mask,
                cmap=cmap,
                annot=annot, 
                fmt=fmt,
                square=True,
                cbar_kws={
                    "shrink": cbar_shrink, 
                    "aspect": cbar_aspect   
                },
                vmin=-1, 
                vmax=1)

    plt.xticks(rotation=0) 
    plt.tight_layout()
    
    return plt.gcf()

In [ ]:
if __name__ == "__main__":

    np.random.seed(42)
    variable=['Chl-a','Three-Band','B1', 'B2','B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8a','B9','B11', 'B12']

    fig = plot_correlation(df, columns=variable,cbar_shrink=0.8, cbar_aspect=30,
                         title='Correlation with Large Colorbar')
    plt.show()